In [1]:
import sys
# Remove the repo root from sys.path so Python imports the installed package
# instead of the local uncompiled source
sys.path = [p for p in sys.path if p not in ("", ".", "d:\\nautilus_trader", "d:/nautilus_trader", "D:\\nautilus_trader", "D:/nautilus_trader")]
from datetime import UTC
from datetime import datetime
from decimal import Decimal
from nautilus_trader.backtest.config import BacktestEngineConfig
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import LoggingConfig
from nautilus_trader.core.datetime import dt_to_unix_nanos
from nautilus_trader.data.config import DataEngineConfig
from nautilus_trader.model import Bar
from nautilus_trader.model import BarType
from nautilus_trader.model import TraderId
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.enums import AccountType
from nautilus_trader.model.enums import OmsType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.model.instruments import Instrument
from nautilus_trader.model.objects import Money
from nautilus_trader.model.objects import Quantity
from nautilus_trader.test_kit.providers import TestInstrumentProvider
from nautilus_trader.trading.strategy import Strategy
import datetime as dt

from nautilus_trader.common.enums import LogColor
from nautilus_trader.core.datetime import unix_nanos_to_dt
from nautilus_trader.model.data import Bar
from nautilus_trader.model.data import BarType
from nautilus_trader.model.enums import OrderSide
from nautilus_trader.model.enums import TimeInForce
from nautilus_trader.model.objects import Quantity
from nautilus_trader.trading.strategy import Strategy


In [2]:
class DemoStrategy(Strategy):
    def __init__(self, input_bartype: BarType):
        super().__init__()

        # Input data
        self.input_bartype = input_bartype
        self.instrument_id = input_bartype.instrument_id
        self.bars_processed = 0

        # Order placed
        self.order_placed = False

        # Start/End time of strategy
        self.start_time = None
        self.end_time = None

    def on_start(self):
        # Remember and log start time of strategy
        self.start_time = dt.datetime.now()
        self.log.info(f"Strategy started at: {self.start_time}")

        # Subscribe to sprimary data
        self.subscribe_bars(self.input_bartype)

    def on_bar(self, bar: Bar):
        self.bars_processed += 1
        self.log.info(
            f"Bar #{self.bars_processed} | Time: {unix_nanos_to_dt(bar.ts_event):%Y-%m-%d %H:%M:%S} | Bar: {bar}",
            color=LogColor.BLUE,
        )

        # Enter: SELL MARKET order (at 3rd bar)
        if not self.order_placed and self.bars_processed == 3:
            order = self.order_factory.market(
                instrument_id=self.instrument_id,
                order_side=OrderSide.SELL,
                quantity=Quantity.from_int(1),  # 1 contract
                time_in_force=TimeInForce.GTC,
            )
            self.submit_order(order)
            self.order_placed = True
            self.log.info(f"Market order placed at {bar.close}", color=LogColor.GREEN)

        # Exit: BUY MARKET order (at 6th bar)
        if self.order_placed and self.bars_processed == 6:
            order = self.order_factory.market(
                instrument_id=self.instrument_id,
                order_side=OrderSide.BUY,
                quantity=Quantity.from_int(1),  # 1 contract
                time_in_force=TimeInForce.GTC,
            )
            self.submit_order(order)
            self.order_placed = True
            self.log.info(f"Market order placed at {bar.close}", color=LogColor.RED)

    def on_stop(self):
        # Remember and log end time of strategy
        self.end_time = dt.datetime.now()
        self.log.info(f"Strategy finished at: {self.end_time}")

        # Log count of processed bars
        self.log.info(f"Total bars processed: {self.bars_processed}")

In [3]:
NANOSECONDS_IN_SECOND = 1_000_000_000

In [4]:
def generate_artificial_bars(instrument: Instrument, bar_type: BarType) -> list[Bar]:
    # Changes between generated bars
    PRICE_CHANGE = instrument.price_increment.as_double() * 10  # 10 ticks
    TIME_CHANGE_NANOS = 60 * NANOSECONDS_IN_SECOND  # 1 minute

    # --------------------------------------------
    # CREATE 1ST BAR
    # --------------------------------------------

    first_bar_time_as_unix_nanos = dt_to_unix_nanos(
        datetime(2024, 2, 1, hour=0, minute=1, second=0, tzinfo=UTC),
    )

    # Add 1st bar
    first_bar = Bar(
        bar_type=bar_type,
        open=instrument.make_price(1.10250),
        high=instrument.make_price(1.10300),
        low=instrument.make_price(1.100000),
        close=instrument.make_price(1.10050),
        volume=Quantity.from_int(999999),
        ts_event=first_bar_time_as_unix_nanos + TIME_CHANGE_NANOS,
        ts_init=first_bar_time_as_unix_nanos + TIME_CHANGE_NANOS,
    )

    generated_bars = [first_bar]
    last_bar = generated_bars[-1]

    # --------------------------------------------
    # CREATE ADDITIONAL BARS
    # --------------------------------------------

    # Add some INCREASING bars
    for _ in range(10):
        last_bar = Bar(
            bar_type=first_bar.bar_type,
            open=instrument.make_price(first_bar.open + PRICE_CHANGE),
            high=instrument.make_price(first_bar.high + PRICE_CHANGE),
            low=instrument.make_price(first_bar.low + PRICE_CHANGE),
            close=instrument.make_price(first_bar.close + PRICE_CHANGE),
            volume=first_bar.volume,
            ts_event=first_bar.ts_event + TIME_CHANGE_NANOS,
            ts_init=first_bar.ts_init + TIME_CHANGE_NANOS,
        )
        generated_bars.append(last_bar)

    # Add some DECREASING bars
    for _ in range(10):
        last_bar = Bar(
            bar_type=first_bar.bar_type,
            open=instrument.make_price(first_bar.open - PRICE_CHANGE),
            high=instrument.make_price(first_bar.high - PRICE_CHANGE),
            low=instrument.make_price(first_bar.low - PRICE_CHANGE),
            close=instrument.make_price(first_bar.close - PRICE_CHANGE),
            volume=first_bar.volume,
            ts_event=first_bar.ts_event + TIME_CHANGE_NANOS,
            ts_init=first_bar.ts_init + TIME_CHANGE_NANOS,
        )
        generated_bars.append(last_bar)

    return generated_bars

In [8]:
from nautilus_trader.analysis.tearsheet import create_tearsheet
from nautilus_trader.analysis.config import TearsheetConfig, TearsheetBarsWithFillsChart
from nautilus_trader.analysis.config import (
    TearsheetConfig,
    TearsheetBarsWithFillsChart,
    TearsheetRunInfoChart,
    TearsheetStatsTableChart,
    TearsheetEquityChart,
    TearsheetDrawdownChart,
    TearsheetMonthlyReturnsChart,
    TearsheetDistributionChart,
    TearsheetRollingSharpeChart,
    TearsheetYearlyReturnsChart
)

def run_backtest():
    # Step 1: Configure and create backtest engine
    engine_config = BacktestEngineConfig(
        trader_id=TraderId("BACKTEST_TRADER-001"),
        # Configure how data will be processed
        data_engine=DataEngineConfig(
            time_bars_interval_type="left-open",
            time_bars_timestamp_on_close=True,
            time_bars_skip_first_non_full_bar=False,
            time_bars_build_with_no_updates=False,  # don't emit aggregated bars, when no source data
            validate_data_sequence=True,
        ),
        # Configure logging
        logging=LoggingConfig(
            log_level="DEBUG",  # set DEBUG log level for console to see loaded bars in logs
        ),
    )
    engine = BacktestEngine(config=engine_config)

    # Step 2: Define exchange and add it to the engine
    VENUE_NAME = "XCME"
    engine.add_venue(
        venue=Venue(VENUE_NAME),
        oms_type=OmsType.NETTING,  # Order Management System type
        account_type=AccountType.MARGIN,  # Type of trading account
        starting_balances=[Money(1_000_000, USD)],  # Initial account balance
        base_currency=USD,  # Base currency for account
        default_leverage=Decimal(1),  # No leverage used for account
    )

    # Step 3: Create instrument definition and add it to the engine
    EURUSD_FUTURE = TestInstrumentProvider.eurusd_future(
        expiry_year=2024,
        expiry_month=3,
        venue_name=VENUE_NAME,
    )
    engine.add_instrument(EURUSD_FUTURE)

    # -------------------------------------------------------
    # PREPARE DATA
    # -------------------------------------------------------

    # Step 4a: Prepare BarType
    EURUSD_1MIN_BARTYPE = BarType.from_str(f"{EURUSD_FUTURE.id}-1-MINUTE-LAST-EXTERNAL")

    # Step 4b: Prepare bar data as list[Bar]
    bars: list[Bar] = generate_artificial_bars(
        instrument=EURUSD_FUTURE,
        bar_type=EURUSD_1MIN_BARTYPE,
    )

    # Step 4c: Add loaded data to the engine
    engine.add_data(bars)

    # ------------------------------------------------------------------------------------------

    # Step 5: Create strategy and add it to the engine
    strategy = DemoStrategy(input_bartype=EURUSD_1MIN_BARTYPE)
    engine.add_strategy(strategy)

    # Step 6: Run engine = Run backtest
    engine.run()
    # Step 7: Generate reports
    order_fill_report = engine.trader.generate_order_fills_report()
    position_report = engine.trader.generate_positions_report()
    account_report = engine.trader.generate_account_report(Venue("XCME"))
    # Step 8: Get backtest result and performance stats
    result = engine.get_result()
    analyzer = engine.portfolio.analyzer
    pnl_stats = analyzer.get_stats_pnls_formatted()
    returns_stats = analyzer.get_stats_returns_formatted()

    tearsheet_path = r"D:\nautilus_trader\reproduce_eg.html"
    create_tearsheet(
        engine=engine,
        output_path=tearsheet_path,
        config=TearsheetConfig(
            theme="nautilus",
            charts=[
                TearsheetRunInfoChart(),
                TearsheetStatsTableChart(),
                TearsheetEquityChart(),
                TearsheetDrawdownChart(),
                TearsheetMonthlyReturnsChart(),
                TearsheetDistributionChart(),
                TearsheetRollingSharpeChart(),
                TearsheetYearlyReturnsChart(),
                TearsheetBarsWithFillsChart(bar_type=EURUSD_1MIN_BARTYPE),
            ],
        ),
    )
    # Step 9: Release system resources
    engine.dispose()

    return order_fill_report, position_report, account_report , result , pnl_stats , returns_stats


In [9]:
order_fill_report, position_report, account_report, result, pnl_stats, returns_stats = run_backtest()

In [10]:
order_fill_report

,trader_id,strategy_id,instrument_id,venue_order_id,position_id,account_id,last_trade_id,type,side,quantity,...,order_list_id,linked_order_ids,parent_order_id,exec_algorithm_id,exec_algorithm_params,exec_spawn_id,tags,init_id,ts_init,ts_last
client_order_id,,,,,,,,,,,,,,,,,,,,,
O-20240201-000300-001-000-1,BACKTEST_TRADER-001,DemoStrategy-000,6EH4.XCME,XCME-1-001,6EH4.XCME-DemoStrategy-000,XCME-001,XCME-1-004,MARKET,SELL,1,...,None,None,None,None,None,None,None,7667cd9c-0c01-4fef-a88a-4ea3ec0bd64e,2024-02-01 00:03:00+00:00,2024-02-01 00:03:00+00:00
O-20240201-000300-001-000-2,BACKTEST_TRADER-001,DemoStrategy-000,6EH4.XCME,XCME-1-002,6EH4.XCME-DemoStrategy-000,XCME-001,XCME-1-008,MARKET,BUY,1,...,None,None,None,None,None,None,None,2e023a17-a750-445d-be4d-464d63f4f6fa,2024-02-01 00:03:00+00:00,2024-02-01 00:03:00+00:00


In [11]:
position_report

,trader_id,strategy_id,instrument_id,account_id,opening_order_id,closing_order_id,entry,side,quantity,peak_qty,...,ts_opened,ts_last,ts_closed,duration_ns,avg_px_open,avg_px_close,commissions,realized_return,realized_pnl,is_snapshot
position_id,,,,,,,,,,,,,,,,,,,,,
6EH4.XCME-DemoStrategy-000,BACKTEST_TRADER-001,DemoStrategy-000,6EH4.XCME,XCME-001,O-20240201-000300-001-000-1,O-20240201-000300-001-000-2,SELL,FLAT,0,1,...,2024-02-01 00:03:00+00:00,1706745780000000000,2024-02-01 00:03:00+00:00,None,1.101,1.101,[0.00 USD],0.0,0.00 USD,False


In [12]:
account_report

,total,locked,free,currency,account_id,account_type,base_currency,margins,reported,info
2024-02-01 00:02:00+00:00,1000000.00,0.00,1000000.00,USD,XCME-001,MARGIN,USD,[],True,{}
2024-02-01 00:03:00+00:00,1000000.00,0.00,1000000.00,USD,XCME-001,MARGIN,USD,[],False,{}
2024-02-01 00:03:00+00:00,1000000.00,0.00,1000000.00,USD,XCME-001,MARGIN,USD,[],False,{}


## Backtest Result Summary

In [13]:
result

BacktestResult(trader_id='BACKTEST_TRADER-001', machine_id='DESKTOP-L51A3P3', run_config_id=None, instance_id='b8ca1964-d5db-4600-8cc7-59167bda600e', run_id='cfdb7e91-d869-4da3-b6b0-ae81343826af', run_started=1774347503156293000, run_finished=1774347503161586000, backtest_start=1706745720000000000, backtest_end=1706745780000000000, elapsed_time=60.0, iterations=21, total_events=4, total_orders=2, total_positions=1, stats_pnls={'USD': {'PnL (total)': 0.0, 'PnL% (total)': 0.0, 'Max Winner': nan, 'Avg Winner': nan, 'Min Winner': nan, 'Min Loser': nan, 'Avg Loser': nan, 'Max Loser': nan, 'Expectancy': 0.0, 'Win Rate': 0.0}}, stats_returns={'Returns Volatility (252 days)': nan, 'Average (Return)': 0.0, 'Average Loss (Return)': nan, 'Average Win (Return)': nan, 'Sharpe Ratio (252 days)': nan, 'Sortino Ratio (252 days)': nan, 'Profit Factor': nan, 'Risk Return Ratio': nan})

In [14]:
import pandas as pd
from dataclasses import asdict

# Flatten the result into a single dict
d = asdict(result)

# Extract stats_pnls per currency into flat keys
pnls = d.pop("stats_pnls")
for currency, stats in pnls.items():
    for k, v in stats.items():
        d[f"{k} ({currency})"] = v

# Extract stats_returns
returns = d.pop("stats_returns")
d.update(returns)

# Display as a single-row DataFrame (transposed for readability)
df = pd.DataFrame([d]).T
df.columns = ["Value"]
df

,Value
trader_id,BACKTEST_TRADER-001
machine_id,DESKTOP-L51A3P3
run_config_id,None
instance_id,b8ca1964-d5db-4600-8cc7-59167bda600e
run_id,cfdb7e91-d869-4da3-b6b0-ae81343826af
run_started,1774347503156293000
run_finished,1774347503161586000
backtest_start,1706745720000000000
backtest_end,1706745780000000000
elapsed_time,60.0


## Returns Statistics

In [15]:
for line in returns_stats:
    print(line)

Returns Volatility (252 days):  nan
Average (Return):               0.0
Average Loss (Return):          nan
Average Win (Return):           nan
Sharpe Ratio (252 days):        nan
Sortino Ratio (252 days):       nan
Profit Factor:                  nan
Risk Return Ratio:              nan


In [17]:
import os
import pandas as pd
from dataclasses import asdict

# Output folder
output_dir = r"D:\nautilus_trader\Results"
os.makedirs(output_dir, exist_ok=True)

# --- Reports from engine ---
# Order Fills
order_fill_report.to_csv(os.path.join(output_dir, "order_fills.csv"))

# Positions
position_report.to_csv(os.path.join(output_dir, "positions.csv"))

# Account (pass your venue)
account_report.to_csv(os.path.join(output_dir, "account.csv"))

# --- BacktestResult to CSV ---
d = asdict(result)
pnls = d.pop("stats_pnls")
for currency, stats in pnls.items():
    for k, v in stats.items():
        d[f"{k} ({currency})"] = v
returns = d.pop("stats_returns")
d.update(returns)

for key in ("run_started", "run_finished", "backtest_start", "backtest_end"):
    if d.get(key):
        d[key] = pd.Timestamp(d[key], unit="ns", tz="UTC").tz_convert("Asia/Kolkata")


result_df = pd.DataFrame([d])
result_df.to_csv(os.path.join(output_dir, "backtest_result.csv"), index=False)



print("Saved to", output_dir)


Saved to D:\nautilus_trader\Results


In [18]:
account_report

,total,locked,free,currency,account_id,account_type,base_currency,margins,reported,info
2024-02-01 00:02:00+00:00,1000000.00,0.00,1000000.00,USD,XCME-001,MARGIN,USD,[],True,{}
2024-02-01 00:03:00+00:00,1000000.00,0.00,1000000.00,USD,XCME-001,MARGIN,USD,[],False,{}
2024-02-01 00:03:00+00:00,1000000.00,0.00,1000000.00,USD,XCME-001,MARGIN,USD,[],False,{}
